# Bonus: Markov Chain Monte Carlo

Here, we present some content that was originally for the second assignment of the class in previous iterations.
Specifically, this covers the topic of Markov chains with some basic examples.
This concept comes up a few times in this class:
RBM sampling via (Block) Gibbs Sampling is one example of Markov chains, and some other iterative models (such as diffusion) later on also run Markov chains.

In [ ]:
import numpy as np
from matplotlib import pyplot as plt

## Part 1: Simple Markov Chains

We first want to practice running a Markov chain for a simple example.
This is easiest when you have a small number of discrete states (numbered 0 to _n_), and a given _transition matrix_ (we call this $A$) giving the probabilities to move from one state to the next.
We define such a matrix below, but you can make your own if you like.
The one requirement is that each colum sums up to 1, as a column gives the probabilities to move _from_ a given state to others.

Now, run a Markov chain:
- Start off with an arbitrary state $x_0$.
- Repeatedly take the column from $A$ that corresponds to the current state $x_i$ (i.e. if the current state is 3, you take the 3rd column).
  Sample a new state $x_{i+1}$ from this probability distribution.
  Do this many times (like, thousands of times) and collect all samples in a list or something similar.
  Sampling can be done via `np.random.choice`.
  You don't need `torch` for this first part at all.
- At the end, plot a histogram of your samples over time. 
This gives you an empirical distribution over the samples. 
Done!
By running your chain you are sampling from the distribution represented by $A$.

We use a small transition matrix with just three states.
But this easily generalizes to an arbitrary number.

In [ ]:
transition_matrix = np.array([[0.2, 0.4, 0.5], [0.6, 0.3, 0.2], [0.2, 0.3, 0.3]])
print(transition_matrix)

n_states = len(transition_matrix)

No we run an actual Markov chain.
We start with an arbitrary initial state and just keep doing random transitions.

In [ ]:
x = 0

all_samples = []
n_steps = 10000
for st in range(n_steps):
    # we just grab the column of the transition matrix that corresponds to the current state
    x = np.random.choice(n_states, size=(), p=transition_matrix[:, x])
    #print(x)
    all_samples.append(x) 

To go a little bit deeper, we consider some theoretical analysis in 
[section 17.3 of the deep learning book](https://www.deeplearningbook.org/).
Try the following (you don't have to read the section):
- Create an arbitrary initial distribution $v_0$, i.e. an $n$-element vector with elements summing to 1.
This represents the _chance_ of being in each state at time step 0, for example if you were running many Markov chains in parallel.
- Multiply the transition matrix with $v_0$, the result is $v_1$.
This gives, for each state, the probability of being in that state at time step 1, i.e. after the first transition from step 0.
Can you work out why this is the case?
- Repeatedly multiply the matrix $A$ with the vector to get the distribution for each time step.
The result should converge very quickly (just a handful of steps) to a vector $v'$. 
This vector represents the probability distribution that this Markov chain (represented by matrix $A$) will converge to!
- Try different starting values of $v_0$. How does this influence $v'$ ?

Does the vector $v'$ match the empirical distribution computed above, when you actually ran a Markov chain? 

We define a random initial distribution `v_start` and repeatedly multiply with the transition matrix. This should converge to the _same_ vector `v_new` every time we run this, even if `v_start` changes. By plotting the difference at each time step, we can see that this process converges very quickly, in just a handful of steps.

In [ ]:
v_start = np.random.rand(n_states, 1)
v_start /= v_start.sum()

n_iterations = 10

v_new = v_start
diffs = []
for st in range(n_iterations):
    v_old = v_new
    v_new = transition_matrix @ v_new
    diffs.append(((v_new - v_old)**2).sum())
    
# we can see that v converges very quickly
plt.plot(np.arange(1, len(diffs)+1), diffs)
plt.xlabel("Steps")
plt.ylabel("Change in v (euclidean norm)")
plt.show()

print("v0")
print(v_start)

print("\n v'")
print(v_new) 

Theory says that the vector this converges to is the eigenvector of the transition matrix with the eigenvalue 1. We can check this by computing the eigenvalues:

In [ ]:
# unless np.eig behaves quirkily, this should be about the same
eigenvector = np.linalg.eig(transition_matrix)[1][:, 0]
print((eigenvector / eigenvector.sum()).real[:, None])

We can compare the empirical distribution (blue bars) by the distribution we computed above (orange stars). These should be near-identical. 

In [ ]:
plt.hist(all_samples, bins=np.linspace(-0.5, n_states-0.5, n_states+1), rwidth=0.9)
plt.plot(v_new * n_steps, "*")
plt.xlabel("State index")
plt.ylabel("Occurrences")
plt.show() 

## Part 2: Gibbs Sampling & Mixing

In most of our use cases, the situation is not as in the example above: 
We don't have a transition distribution given and just start running a Markov chain on it to get some marginal distribution.
Instead, we have a _desired target distribution_ (e.g. our model distribution) and need to figure out how to get there, i.e. how to sample from it.

Let's try to sample from a mixture of Gaussians via Gibbs sampling.
- Set up a distribution. 
Gibbs sampling makes use of conditional distributions.
In order to be able to do any conditioning, the full distribution needs to be multivariate. 
For simplicity (and easy plotting), we stick to a 2D distribution.
Also, we want at least two components;
these can be simple independent Gaussians.
- Start with an arbitrary initial sample (e.g. a vector of 0s). 
Now, repeatedly do the following: 
Sample a new value for _x_ given the value for _y_. 
Then, sample a new value for _y_ given the (new!) value for _x_. 
Since we now have sampled new values for both dimensions, we have essentially taken a new sample of our 2D distribution.
- We use `torch.distributions` to build the distributions, specifically a `MixtureSameFamily` of `Normal` distributions.
- The hardest part is figuring out how to take a conditional sample. 
It turns out that in this case, where $p(x,y)$ is a mixture of Gaussians, the conditional distribution $p(x|y)$ (as well
as the other way around) is a mixture of Gaussians as well! 
You can find a derivation [here](https://stats.stackexchange.com/questions/348941/general-conditional-distributions-for-multivariate-gaussian-mixtures).
Being able to translate such formulae into code is important!
However, the derivation linked above is for a fairly general one, and can be simplified considerably in our case.
By using independent Gaussians as components, the only thing that actually changes in the conditional distribution are 
the mixture coefficients!
- We use `torch.distributions` to sample from the conditional, one-dimensional, mixtures of Gaussians as mentioned 
above. 
You might say: 
Why not just sample from a two-dimensional mixture of Gaussians in the first place? 
The reason we don't do this is that this is a contrived example to show the concept of Gibbs sampling. ;)

We need to collect a reasonable number of samples (1000 or more) and plot both the target distribution (mixture of
Gaussians) and your samples. 
Do the samples reflect the distribution well? 
In particular are both modes of the Gaussian mixture covered equally? 
You can decide on this visually and/or using statistics. 
Also, experiment with different locations/scales for the Gaussians.
That is, move the components further apart or closer together and repeat the sampling process each time.
The quality of the samples should vary dramatically based on the distance between components!

You can run this with different values for the `location` parameter. Increasing this moves the two distributions further apart. Alternatively, keeping `location` constant but reducing `scale` has an equivalent effect.

In [ ]:
import torch
td = torch.distributions

location = 3.
scale = 1.
sample = torch.tensor([0.,0.])  # initial sample -- arbitrary
# "balanced" Gaussian mixture
basic_distribution = td.MixtureSameFamily(
    td.Categorical(probs=torch.tensor([0.5, 0.5])), 
    td.MultivariateNormal(loc=torch.tensor([[-location, -location], [location, location]]),
                          covariance_matrix=scale*torch.eye(2)))

n_steps = 1000
samples = [sample]
for step in range(n_steps):
    sample = sample.clone()
    # sample new x1 (given OLD x2)
    mixing_score1 = torch.exp(td.Normal(loc=-location, scale=scale).log_prob(sample[1]))
    mixing_score2 = torch.exp(td.Normal(loc=location, scale=scale).log_prob(sample[1]))
    pi1 = mixing_score1 / (mixing_score1 + mixing_score2)
    pi2 = mixing_score2 / (mixing_score1 + mixing_score2)
    # the normal distributions stay the same (same mean)
    # but the mixture coefficients are different.
    # note that we are now using a 1D normal because we are only sampling one variable
    conditional_distribution = td.MixtureSameFamily(
        td.Categorical(probs=torch.tensor([pi1, pi2])),
        td.Normal(loc=torch.tensor([-location, location]),
                  scale=scale))
    sample[0] = conditional_distribution.sample()
    
    # sample new x2 given the NEW x1
    mixing_score1 = torch.exp(td.Normal(loc=-location, scale=scale).log_prob(sample[0]))
    mixing_score2 = torch.exp(td.Normal(loc=location, scale=scale).log_prob(sample[0]))
    pi1 = mixing_score1 / (mixing_score1 + mixing_score2)
    pi2 = mixing_score2 / (mixing_score1 + mixing_score2)
    conditional_distribution = td.MixtureSameFamily(
        td.Categorical(probs=torch.tensor([pi1, pi2])),
        td.Normal(loc=torch.tensor([-location, location]),
                  scale=scale))
    sample[1] = conditional_distribution.sample()
    samples.append(sample)
samples = np.array(samples) 

- Contour plot shows the real distribution.
- Black dots show samples.
- Lines show the steps between samples.
- The earliest samples have lines in orange, and over the course of sampling, the line becomes more blue.

In [ ]:
lim = 2*location if location >= 2 else 4
grid_points = np.linspace(-lim, lim, 1000)
grid_x, grid_y = np.meshgrid(grid_points, grid_points)
grid = np.stack([grid_x, grid_y], axis=-1) 

plt.figure(figsize=(10, 10))
plt.contour(grid_x, grid_y, np.exp(basic_distribution.log_prob(torch.tensor(grid))))

color_gradient = np.linspace(0,1,len(samples))
for i in range(len(samples)-1):
    plt.plot(samples[i:i+2, 0],samples[i:i+2, 1], color=(1-color_gradient[i],
                                                         0.5,
                                                         color_gradient[i]), alpha=0.3)

plt.plot(samples[:,0], samples[:,1], ".", c="black", alpha=0.5)
plt.show() 

A heat map gives a better view of whether both modes have been sampled evenly.

Bonus: This could be done more properly via statistical testing.
That is, check whether the amount of samples per quadrant is significantly different from what we would expect (should be about equal) 

In [ ]:
plt.figure(figsize=(10,10))
plt.hist2d(samples[:,0], samples[:,1], bins=30)
plt.show()

Or we could count the occurrences of samples per quadrant:

In [ ]:
quadrant1 = np.logical_and(samples[:, 0] >= 0, samples[:, 1] >= 0).sum()
#quadrant2 = np.logical_and(samples[:, 0] >= 0, samples[:, 1] < 0).sum()
#quadrant3 = np.logical_and(samples[:, 0] < 0, samples[:, 1] >= 0).sum()
quadrant4 = np.logical_and(samples[:, 0] < 0, samples[:, 1] < 0).sum()

print(quadrant1, quadrant4)  # should be approximately equal if sampling works well